In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor, ViTModel
from PIL import Image
import mplfinance as mpf
import os
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler   
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
import matplotlib.pyplot as plt

C:\Users\sam4s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the merged file with unnamed first column as index
df = pd.read_csv("nvda_prices_sentiment_vit.csv", 
                 index_col=[0],         
                 parse_dates=True, 
                 dayfirst=True)

# Add technical indicators - Simple Moving Averages (SMA) and Relative Strength Index (RSI)
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_30'] = df['Close'].rolling(window=30).mean()

# RSI (14-period)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Drop rows with NaN from rolling calculations
df = df.dropna()

print("Indicators added. New columns:", ['SMA_10', 'SMA_30', 'RSI'])

print("=== Merged Data Confirmed ===")
print("Shape:", df.shape)
print("\nIndex type:", type(df.index))
print("First 5 index dates:", df.index[:5])
print("\nColumns (first 10 + last 5):")
print(df.columns[:10].tolist() + ["..."] + df.columns[-5:].tolist())
print("\nSample rows (last 5):")
print(df.tail(5)[['Close', 'finbert_sentiment'] + 
                 [col for col in df.columns if col.startswith('vit_')][:3]])

Indicators added. New columns: ['SMA_10', 'SMA_30', 'RSI']
Shape after dropna: (971, 778)
=== Merged Data Confirmed ===
Shape: (971, 778)

Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
First 5 index dates: DatetimeIndex(['2022-02-04', '2022-02-07', '2022-02-08', '2022-02-09',
               '2022-02-10'],
              dtype='datetime64[ns]', freq=None)

Columns (first 10 + last 5):
['key_0', 'Close', 'High', 'Low', 'Open', 'Volume', 'finbert_sentiment', 'vit_dim_0', 'vit_dim_1', 'vit_dim_2', '...', 'vit_dim_766', 'vit_dim_767', 'SMA_10', 'SMA_30', 'RSI']

Sample rows (last 5):
                 Close  finbert_sentiment  vit_dim_0  vit_dim_1  vit_dim_2
2025-12-11  180.920197                0.0   0.195352  -0.440696   0.217055
2025-12-12  175.010529                0.0   0.159925  -0.282702   0.469230
2025-12-15  176.280457                0.0   0.247811  -0.377223   0.789828
2025-12-16  177.710388                0.0   0.330134  -0.345409   0.637049
2025-12-17  170.9307

C:\Users\sam4s\AppData\Local\Temp\ipykernel_892\1819386739.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv("nvda_prices_sentiment_vit.csv",


In [3]:
# Basic stats on sentiment and a few ViT dims
print(df['finbert_sentiment'].describe())
print("\nExample ViT stats (first 3 dims):")
print(df[[f'vit_dim_{i}' for i in range(3)]].describe())

count    1000.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: finbert_sentiment, dtype: float64

Example ViT stats (first 3 dims):
         vit_dim_0    vit_dim_1    vit_dim_2
count  1000.000000  1000.000000  1000.000000
mean      0.037752    -0.139375     0.348387
std       0.304352     0.271086     0.322886
min      -0.708345    -1.008653    -0.753861
25%      -0.192475    -0.326938     0.171115
50%       0.042139    -0.152438     0.378523
75%       0.267116     0.066087     0.543254
max       0.815085     0.832584     1.424769


In [4]:
# Quick check on the presence of ViT features and sentiment
print("Full shape:", df.shape)
print("\nNumber of ViT columns:", len([col for col in df.columns if col.startswith('vit_dim_')]))
print("Sample ViT columns:", [col for col in df.columns if col.startswith('vit_dim_')][:5])
print("\nSentiment column present?", 'finbert_sentiment' in df.columns)
print("Total features expected:", 5 + 1 + 3 + 768)  # Open High Low Close Volume + sentiment + 3 indicators + 768 ViT

Full shape: (1000, 775)

Number of ViT columns: 768
Sample ViT columns: ['vit_dim_0', 'vit_dim_1', 'vit_dim_2', 'vit_dim_3', 'vit_dim_4']

Sentiment column present? True
Total features expected: 777


Training the full multimodal LSTM

In [ ]:
df = pd.read_csv("nvda_prices_sentiment_vit.csv", 
                 index_col=[0],          
                 parse_dates=True, 
                 dayfirst=True)

print("Loaded shape:", df.shape)
print("Columns right after load:", df.columns.tolist())

df['next_close'] = df['Close'].shift(-1)           # tomorrow's close
df['direction'] = (df['next_close'] > df['Close']).astype(int)  # 1 = up

# Drop the last row (no next_close for it)
df = df.dropna(subset=['next_close', 'direction'])

print("Targets added. New columns:", ['next_close', 'direction'])
print(df.tail(3)[['Close', 'next_close', 'direction']])

features = ['Open', 'High', 'Low', 'Close', 'Volume', 
            'finbert_sentiment', 'SMA_10', 'SMA_30', 'RSI'] + \
           [f"vit_dim_{i}" for i in range(768)]

print("Features defined. Total:", len(features))

Loaded shape: (1000, 775)
Columns right after load: ['key_0', 'Close', 'High', 'Low', 'Open', 'Volume', 'finbert_sentiment', 'vit_dim_0', 'vit_dim_1', 'vit_dim_2', 'vit_dim_3', 'vit_dim_4', 'vit_dim_5', 'vit_dim_6', 'vit_dim_7', 'vit_dim_8', 'vit_dim_9', 'vit_dim_10', 'vit_dim_11', 'vit_dim_12', 'vit_dim_13', 'vit_dim_14', 'vit_dim_15', 'vit_dim_16', 'vit_dim_17', 'vit_dim_18', 'vit_dim_19', 'vit_dim_20', 'vit_dim_21', 'vit_dim_22', 'vit_dim_23', 'vit_dim_24', 'vit_dim_25', 'vit_dim_26', 'vit_dim_27', 'vit_dim_28', 'vit_dim_29', 'vit_dim_30', 'vit_dim_31', 'vit_dim_32', 'vit_dim_33', 'vit_dim_34', 'vit_dim_35', 'vit_dim_36', 'vit_dim_37', 'vit_dim_38', 'vit_dim_39', 'vit_dim_40', 'vit_dim_41', 'vit_dim_42', 'vit_dim_43', 'vit_dim_44', 'vit_dim_45', 'vit_dim_46', 'vit_dim_47', 'vit_dim_48', 'vit_dim_49', 'vit_dim_50', 'vit_dim_51', 'vit_dim_52', 'vit_dim_53', 'vit_dim_54', 'vit_dim_55', 'vit_dim_56', 'vit_dim_57', 'vit_dim_58', 'vit_dim_59', 'vit_dim_60', 'vit_dim_61', 'vit_dim_62', 'vi

C:\Users\sam4s\AppData\Local\Temp\ipykernel_892\3518798576.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv("nvda_prices_sentiment_vit.csv",


Create the Final File for Multimodel Training (price+sentiment+vit)

In [13]:
input_file = "nvda_prices_sentiment_vit.csv" 
df = pd.read_csv(input_file, 
                 index_col=[0], 
                 parse_dates=True, 
                 dayfirst=True)

print("Loaded shape:", df.shape)
print("Columns before adding:", df.columns.tolist())

# Add ALL missing indicators & targets
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_30'] = df['Close'].rolling(window=30).mean()

# RSI
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Targets
df['next_close'] = df['Close'].shift(-1)
df['direction'] = (df['next_close'] > df['Close']).astype(int)

df = df.dropna()

final_file = "nvda_complete_multimodal.csv"
df.to_csv(final_file)
print(f"\nCOMPLETE file saved to:\n{final_file}")
print("Final shape:", df.shape)
print("Final columns:", df.columns.tolist())
print("\nLast 5 rows preview:")
print(df.tail(5)[['Close', 'finbert_sentiment', 'SMA_10', 'SMA_30', 'RSI', 'next_close', 'direction']])

C:\Users\sam4s\AppData\Local\Temp\ipykernel_892\2382123503.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv(input_file,


Loaded shape: (1000, 775)
Columns before adding: ['key_0', 'Close', 'High', 'Low', 'Open', 'Volume', 'finbert_sentiment', 'vit_dim_0', 'vit_dim_1', 'vit_dim_2', 'vit_dim_3', 'vit_dim_4', 'vit_dim_5', 'vit_dim_6', 'vit_dim_7', 'vit_dim_8', 'vit_dim_9', 'vit_dim_10', 'vit_dim_11', 'vit_dim_12', 'vit_dim_13', 'vit_dim_14', 'vit_dim_15', 'vit_dim_16', 'vit_dim_17', 'vit_dim_18', 'vit_dim_19', 'vit_dim_20', 'vit_dim_21', 'vit_dim_22', 'vit_dim_23', 'vit_dim_24', 'vit_dim_25', 'vit_dim_26', 'vit_dim_27', 'vit_dim_28', 'vit_dim_29', 'vit_dim_30', 'vit_dim_31', 'vit_dim_32', 'vit_dim_33', 'vit_dim_34', 'vit_dim_35', 'vit_dim_36', 'vit_dim_37', 'vit_dim_38', 'vit_dim_39', 'vit_dim_40', 'vit_dim_41', 'vit_dim_42', 'vit_dim_43', 'vit_dim_44', 'vit_dim_45', 'vit_dim_46', 'vit_dim_47', 'vit_dim_48', 'vit_dim_49', 'vit_dim_50', 'vit_dim_51', 'vit_dim_52', 'vit_dim_53', 'vit_dim_54', 'vit_dim_55', 'vit_dim_56', 'vit_dim_57', 'vit_dim_58', 'vit_dim_59', 'vit_dim_60', 'vit_dim_61', 'vit_dim_62', 'vit_d